# Clasificación de Piso en el Dataset UJIIndoorLoc

---

## Introducción

En este notebook se implementa un flujo completo de procesamiento y análisis para la clasificación del **piso** en un entorno interior utilizando el dataset **UJIIndoorLoc**. Este conjunto de datos contiene mediciones de señales WiFi recopiladas en distintas ubicaciones de un edificio, con información sobre coordenadas, piso, usuario, hora, entre otros.

En esta tarea nos enfocaremos en predecir el **piso** en el que se encuentra un dispositivo, considerando únicamente las muestras etiquetadas con valores válidos para dicha variable. Se tratará como un problema de clasificación multiclase (planta baja, primer piso, segundo piso).

## Objetivos

- **Cargar y explorar** el conjunto de datos UJIIndoorLoc.
- **Preparar** los datos seleccionando las características relevantes y el target (`FLOOR`).
- **Dividir** el dataset en entrenamiento y validación (80/20).
- **Entrenar y optimizar** clasificadores basados en seis algoritmos:
  - K-Nearest Neighbors (KNN)
  - Gaussian Naive Bayes
  - Regresión Logística
  - Árboles de Decisión
  - Support Vector Machines (SVM)
  - Random Forest
- **Seleccionar hiperparámetros óptimos** para cada modelo utilizando validación cruzada (5-fold), empleando estrategias como **Grid Search**, **Randomized Search**, o **Bayesian Optimization** según el algoritmo.
- **Comparar el desempeño** de los modelos sobre el conjunto de validación, usando métricas como *accuracy*, *precision*, *recall*, y *F1-score*.
- **Determinar el mejor clasificador** para esta tarea, junto con sus hiperparámetros óptimos.

Este ejercicio permite no solo evaluar la capacidad predictiva de distintos algoritmos clásicos de clasificación, sino también desarrollar buenas prácticas en validación de modelos y selección de hiperparámetros en contextos del mundo real.

---

## Descripción del Dataset

El dataset utilizado en este análisis es el **UJIIndoorLoc Dataset**, ampliamente utilizado para tareas de localización en interiores a partir de señales WiFi. Está disponible públicamente en la UCI Machine Learning Repository y ha sido recopilado en un entorno real de un edificio universitario.

Cada muestra corresponde a una observación realizada por un dispositivo móvil, donde se registran las intensidades de señal (RSSI) de más de 500 puntos de acceso WiFi disponibles en el entorno. Además, cada fila contiene información contextual como la ubicación real del dispositivo (coordenadas X e Y), el piso, el edificio, el identificador del usuario, y la marca temporal.

El objetivo en esta tarea es predecir el **piso** (`FLOOR`) en el que se encontraba el dispositivo en el momento de la medición, considerando únicamente las características numéricas provenientes de las señales WiFi.

### Estructura del dataset

- **Número de muestras**: ~20,000
- **Número de características**: 520
  - 520 columnas con valores de intensidad de señal WiFi (`WAP001` a `WAP520`)
- **Variable objetivo**: `FLOOR` (variable categórica con múltiples clases, usualmente entre 0 y 4)

### Columnas relevantes

- `WAP001`, `WAP002`, ..., `WAP520`: niveles de señal recibida desde cada punto de acceso WiFi (valores entre -104 y 0, o 100 si no se detectó).
- `FLOOR`: clase objetivo a predecir (nivel del edificio).
- (Otras columnas como `BUILDINGID`, `SPACEID`, `USERID`, `TIMESTAMP`, etc., pueden ser ignoradas o utilizadas en análisis complementarios).

### Contexto del problema

La localización en interiores es un problema complejo en el que tecnologías como el GPS no funcionan adecuadamente. Los sistemas basados en WiFi han demostrado ser una alternativa efectiva para estimar la ubicación de usuarios en edificios. Poder predecir automáticamente el piso en el que se encuentra una persona puede mejorar aplicaciones de navegación en interiores, accesibilidad, gestión de emergencias y servicios personalizados. Este tipo de problemas es típicamente abordado mediante algoritmos de clasificación multiclase.


### Estrategia de evaluación

En este análisis seguiremos una metodología rigurosa para garantizar la validez de los resultados:

1. **Dataset de entrenamiento**: Se utilizará exclusivamente para el desarrollo, entrenamiento y optimización de hiperparámetros de todos los modelos. Este conjunto será dividido internamente en subconjuntos de entrenamiento y validación (80/20) para la selección de hiperparámetros mediante validación cruzada.

2. **Dataset de prueba**: Se reservará únicamente para la **evaluación final** de los modelos ya optimizados. Este conjunto **no debe ser utilizado** durante el proceso de selección de hiperparámetros, ajuste de modelos o toma de decisiones sobre la arquitectura, ya que esto introduciría sesgo y comprometería la capacidad de generalización estimada.

3. **Validación cruzada**: Para la optimización de hiperparámetros se empleará validación cruzada 5-fold sobre el conjunto de entrenamiento, lo que permitirá una estimación robusta del rendimiento sin contaminar los datos de prueba.

Esta separación estricta entre datos de desarrollo y evaluación final es fundamental para obtener una estimación realista del rendimiento que los modelos tendrían en un escenario de producción con datos completamente nuevos.

---


## Paso 1: Cargar y explorar el dataset

**Instrucciones:**
- Descarga el dataset **UJIIndoorLoc** desde la UCI Machine Learning Repository o utiliza la versión proporcionada en el repositorio del curso (por ejemplo: `datasets\UJIIndoorLoc\trainingData.csv`).
- Carga el dataset utilizando `pandas`.
- Muestra las primeras filas del dataset utilizando `df.head()`.
- Imprime el número total de muestras (filas) y características (columnas).
- Verifica cuántas clases distintas hay en la variable objetivo `FLOOR` y cuántas muestras tiene cada clase (`df['FLOOR'].value_counts()`).


In [1]:
import pandas as pd
from pathlib import Path

# Ruta al CSV 
train_path = Path("dataset/trainingData.csv") if Path("dataset/trainingData.csv").exists() else Path("dataset/trainingData")

# Carga (detección automática de separador)
df = pd.read_csv(train_path, sep=None, engine="python")

# Vistazo rápido
print("Shape (filas, columnas):", df.shape)
display(df.head())

# Conteo de la variable objetivo FLOOR
floor_col = next(c for c in df.columns if c.upper() == "FLOOR")
print("\nClases distintas en FLOOR:", sorted(df[floor_col].unique()))
print("\nMuestras por clase:")
display(df[floor_col].value_counts().sort_index())


Shape (filas, columnas): (19937, 529)


,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,...,WAP520,LONGITUDE,LATITUDE,FLOOR,BUILDINGID,SPACEID,RELATIVEPOSITION,USERID,PHONEID,TIMESTAMP
0,100,100,100,100,100,100,100,100,100,100,...,100,-7541.2643,4.864921e+06,2,1,106,2,2,23,1371713733
1,100,100,100,100,100,100,100,100,100,100,...,100,-7536.6212,4.864934e+06,2,1,106,2,2,23,1371713691
2,100,100,100,100,100,100,100,-97,100,100,...,100,-7519.1524,4.864950e+06,2,1,103,2,2,23,1371714095
3,100,100,100,100,100,100,100,100,100,100,...,100,-7524.5704,4.864934e+06,2,1,102,2,2,23,1371713807
4,100,100,100,100,100,100,100,100,100,100,...,100,-7632.1436,4.864982e+06,0,0,122,2,11,13,1369909710



Clases distintas en FLOOR: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Muestras por clase:


FLOOR
0    4369
1    5002
2    4416
3    5048
4    1102
Name: count, dtype: int64

---

## Paso 2: Preparar los datos

**Instrucciones:**

- Elimina las columnas que no son relevantes para la tarea de clasificación del piso:
  - `LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`
- Conserva únicamente:
  - Las columnas `WAP001` a `WAP520` como características (RSSI de puntos de acceso WiFi).
  - La columna `FLOOR` como variable objetivo.
- Verifica si existen valores atípicos o valores inválidos en las señales WiFi (por ejemplo: valores constantes como 100 o -110 que suelen indicar ausencia de señal).
- Separa el conjunto de datos en:
  - `X`: matriz de características (todas las columnas `WAP`)
  - `y`: vector objetivo (`FLOOR`)


In [2]:
# --- Paso 2: preparar los datos ---

import pandas as pd

# 1) Eliminar columnas no relevantes
drop_cols = ["LONGITUDE","LATITUDE","SPACEID","RELATIVEPOSITION",
             "USERID","PHONEID","TIMESTAMP"]
drop_cols = [c for c in drop_cols if c in df.columns]   # por si alguna no existe
df2 = df.drop(columns=drop_cols)

# 2) Detectar columnas WAP y la columna objetivo FLOOR
wap_cols = [c for c in df2.columns if c.upper().startswith("WAP")]
target_col = next(c for c in df2.columns if c.upper() == "FLOOR")

# 3) Asegurar numérico en WAPs
df2[wap_cols] = df2[wap_cols].apply(pd.to_numeric, errors="coerce")

# 4) Chequeo rápido de valores inválidos típicos
n_100 = (df2[wap_cols] == 100).sum().sum()
n_m110 = (df2[wap_cols] == -110).sum().sum()
print(f"Conteo de valores especiales -> 100: {n_100:,} | -110: {n_m110:,}")

# 5) Limpieza mínima:
#    - Reemplazar 100 (ausencia de señal en algunas versiones) por -110
#    - Limitar el rango a [-110, 0]
#    - Rellenar NaN con -110
df2[wap_cols] = (
    df2[wap_cols]
      .replace(100, -110)
      .clip(lower=-110, upper=0)
      .fillna(-110)
)

# 6) Separar en X (features) e y (target)
X = df2[wap_cols].copy()
y = df2[target_col].astype(int).copy()

print("X shape:", X.shape, "| y shape:", y.shape)
print("Rango real de RSSI tras limpieza:",
      int(X.min().min()), "a", int(X.max().max()))
print("Clases FLOOR:", sorted(y.unique()))


Conteo de valores especiales -> 100: 10,008,477 | -110: 0
X shape: (19937, 520) | y shape: (19937,)
Rango real de RSSI tras limpieza: -110 a 0
Clases FLOOR: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


--- 

## Paso 3: Preprocesamiento de las señales WiFi

**Contexto:**

Las columnas `WAP001` a `WAP520` representan la intensidad de la señal (RSSI) recibida desde distintos puntos de acceso WiFi. Los valores típicos de RSSI están en una escala negativa, donde:

- Valores cercanos a **0 dBm** indican señal fuerte.
- Valores cercanos a **-100 dBm** indican señal débil o casi ausente.
- Un valor de **100** en este dataset representa una señal **no detectada**, es decir, el punto de acceso no fue visto por el dispositivo en ese instante.

**Instrucciones:**

- Para facilitar el procesamiento y tratar la ausencia de señal de forma coherente, se recomienda mapear todos los valores **100** a **-100**, que semánticamente representa *ausencia de señal detectable*.
- Esto unifica el rango de valores y evita que 100 (un valor artificial) afecte negativamente la escala de los algoritmos.

**Pasos sugeridos:**

- Reemplaza todos los valores `100` por `-100` en las columnas `WAP001` a `WAP520`:
  ```python
  X[X == 100] = -100


In [3]:
# --- Paso 3: preprocesamiento de señales WiFi (WAP001–WAP520) ---

import pandas as pd

# 1) Detectar columnas WAP (por si X tiene otras)
wap_cols = [c for c in X.columns if str(c).upper().startswith("WAP")]

# 2) Asegurar tipo numérico
X[wap_cols] = X[wap_cols].apply(pd.to_numeric, errors="coerce")

# 3) Unificar ausencia de señal:
#    - Si venías del paso 2 con -110, también lo llevamos a -100
X[wap_cols] = X[wap_cols].replace({100: -100, -110: -100})

# 4) Chequeo rápido
print("Rango RSSI después del mapeo:",
      int(X[wap_cols].min().min()), "a", int(X[wap_cols].max().max()))
print("Total de -100 (ausencia de señal):",
      int((X[wap_cols] == -100).sum().sum()))


Rango RSSI después del mapeo: -104 a 0
Total de -100 (ausencia de señal): 10008716


--- 

## Paso 4: Entrenamiento y optimización de hiperparámetros

**Objetivo:**

Entrenar y comparar distintos clasificadores para predecir correctamente el piso (`FLOOR`) y encontrar los mejores hiperparámetros para cada uno mediante validación cruzada.

**Clasificadores a evaluar:**

- K-Nearest Neighbors (KNN)
- Gaussian Naive Bayes
- Regresión Logística
- Árboles de Decisión
- Support Vector Machines (SVM)
- Random Forest

**Procedimiento:**

1. Divide el dataset en conjunto de **entrenamiento** (80%) y **validación** (20%) usando `train_test_split` con `stratify=y`.
2. Para cada clasificador:
   - Define el espacio de búsqueda de hiperparámetros.
   - Usa **validación cruzada 5-fold** sobre el conjunto de entrenamiento para seleccionar los mejores hiperparámetros.
   - Emplea una estrategia de búsqueda adecuada:
     - **GridSearchCV**: búsqueda exhaustiva (ideal para espacios pequeños).
     - **RandomizedSearchCV**: búsqueda aleatoria (más eficiente con espacios amplios).
     - **Bayesian Optimization** (opcional): para búsquedas más inteligentes, usando librerías como `optuna` o `skopt`.
3. Guarda el mejor modelo encontrado para cada clasificador con su configuración óptima.



In [4]:
# create the training and validation sets
# === Imports y split ===
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score
)

SEED = 42

# Asegúrate de tener X, y preparados desde los pasos 2–3 (solo WAP001..WAP520 y FLOOR, con 100 -> -100)
assert 'X' in globals() and 'y' in globals(), "Asegúrate de haber ejecutado los pasos 2 y 3 (X, y)."

# 80/20 estratificado
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

# 5-fold estratificado para CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# para acumular resultados
results = []

# helper genérico para todos los modelos (GridSearchCV por defecto)
def run_grid(name, pipe, param_grid, scoring='accuracy'):
    gs = GridSearchCV(
        pipe, param_grid, cv=cv, scoring=scoring, n_jobs=-1, verbose=0
    )
    gs.fit(X_tr, y_tr)
    y_pred = gs.best_estimator_.predict(X_te)

    acc  = accuracy_score(y_te, y_pred)
    bacc = balanced_accuracy_score(y_te, y_pred)
    f1m  = f1_score(y_te, y_pred, average="macro")

    print(f"{name:15s} | best_cv={gs.best_score_:.4f} | "
          f"test_acc={acc:.4f} | test_bal_acc={bacc:.4f} | f1_macro={f1m:.4f}")

    results.append({
        "modelo": name,
        "cv_acc": gs.best_score_,
        "test_acc": acc,
        "test_bal_acc": bacc,
        "test_f1_macro": f1m,
        "best_params": gs.best_params_,
        "best_estimator_": gs.best_estimator_
    })
    return gs


In [5]:
# train and optimize KNN
from sklearn.neighbors import KNeighborsClassifier

# Nota: si tu X fuera sparse, usa StandardScaler(with_mean=False)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(n_jobs=-1))
])

param_grid = {
    "clf__n_neighbors": [3, 5, 7, 9, 11],
    "clf__weights": ["uniform", "distance"],
    "clf__p": [1, 2],  # 1=Manhattan, 2=Euclídea
}

gs_knn = run_grid("KNN", pipe, param_grid)


KNN             | best_cv=0.9940 | test_acc=0.9962 | test_bal_acc=0.9967 | f1_macro=0.9968


In [6]:
# train and optimize Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB

pipe = Pipeline([
    ("clf", GaussianNB())
])

param_grid = {
    "clf__var_smoothing": [1e-9, 1e-8, 1e-7]
}

gs_gnb = run_grid("GaussianNB", pipe, param_grid)

GaussianNB      | best_cv=0.5907 | test_acc=0.5785 | test_bal_acc=0.6463 | f1_macro=0.5618


In [7]:
# train and optimize Logistic Regression
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

param_grid = {
    "clf__C": [0.1, 1, 10],
    "clf__solver": ["lbfgs", "liblinear"],
}

gs_lr = run_grid("LogReg", pipe, param_grid)


LogReg          | best_cv=0.9931 | test_acc=0.9922 | test_bal_acc=0.9929 | f1_macro=0.9927


In [8]:
# train and optimize decision tree
from sklearn.tree import DecisionTreeClassifier

pipe = Pipeline([
    ("clf", DecisionTreeClassifier(
        random_state=SEED, class_weight="balanced"
    ))
])

param_grid = {
    "clf__max_depth": [10, 20, 30, None],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 5],
}

gs_dt = run_grid("DecisionTree", pipe, param_grid)


DecisionTree    | best_cv=0.9656 | test_acc=0.9712 | test_bal_acc=0.9758 | f1_macro=0.9736


In [9]:
# train and optimize Support Vector Machine
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV

pipe = Pipeline([
    ("scaler", StandardScaler()),   # si X es sparse => StandardScaler(with_mean=False)
    ("clf", SVC(kernel="rbf", cache_size=1000))
])

param_grid = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__gamma": ["scale", 0.01, 0.001],
    "clf__class_weight": [None, "balanced"],   # <- mejora opcional
}

svm_halving = HalvingGridSearchCV(
    pipe, param_grid, cv=cv, n_jobs=-1, scoring="accuracy",
    factor=3, verbose=0
)
svm_halving.fit(X_tr, y_tr)

y_pred = svm_halving.best_estimator_.predict(X_te)
acc  = accuracy_score(y_te, y_pred)
bacc = balanced_accuracy_score(y_te, y_pred)
f1m  = f1_score(y_te, y_pred, average="macro")

print("SVM (rbf/halving) best:", svm_halving.best_params_,
      "| cv_acc:", svm_halving.best_score_)
print("test_acc:", acc, "| test_bal_acc:", bacc, "| f1_macro:", f1m)

results.append({
    "modelo": "SVM (rbf/halving)",
    "best_params": svm_halving.best_params_,
    "best_cv": svm_halving.best_score_,
    "cv_acc": svm_halving.best_score_,
    "test_acc": acc,
    "test_bal_acc": bacc,
    "test_f1_macro": f1m,
    "best_estimator_": svm_halving.best_estimator_
})


SVM (rbf/halving) best: {'clf__C': 10, 'clf__class_weight': None, 'clf__gamma': 0.001} | cv_acc: 0.9941043809236032
test_acc: 0.9949849548645938 | test_bal_acc: 0.9955824990359847 | f1_macro: 0.9957476468037754


In [10]:
# train and optimize Random Forest
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
import joblib
import re

pipe = Pipeline([
    ("clf", RandomForestClassifier(
        random_state=SEED, n_jobs=-1, class_weight="balanced"
    ))
])

param_grid = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [None, 20, 40],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
}

gs_rf = run_grid("RandomForest", pipe, param_grid)

# --- Resumen ordenado ---
summary = pd.DataFrame([
    {k: v for k, v in r.items() if k != "best_estimator_"} for r in results
]).sort_values("test_acc", ascending=False).reset_index(drop=True)

display(summary)

# --- Persistir el mejor modelo ---
best_idx = summary["test_acc"].idxmax()
best_model_name = summary.loc[best_idx, "modelo"]
best_estimator = results[best_idx]["best_estimator_"]
joblib.dump(best_estimator, "best_model.pkl")
print(f"✅ Mejor modelo guardado: {best_model_name} -> best_model.pkl")

# --- (Opcional) Evaluación sobre validationData.csv ---
# Repite el mismo preprocesamiento mínimo: usar mismas columnas WAP y mapear 100 -> -100
val_path = r"C:\IA_RigobertoZelayandia\CLase-IA\PROYECTO 2\dataset\validationData.csv"
train_path = r"C:\IA_RigobertoZelayandia\CLase-IA\PROYECTO 2\dataset\trainingData.csv"

try:
    df_val = pd.read_csv(val_path, low_memory=False)
    df_tr_cols = pd.read_csv(train_path, nrows=5, low_memory=False)  # solo para tomar las columnas

    # columnas WAP ordenadas como WAP001..WAP520
    wap_cols = [c for c in df_tr_cols.columns if re.match(r"^WAP\d+$", c)]
    wap_cols = sorted(wap_cols, key=lambda s: int(s[3:]))

    # asegurar columnas y orden
    X_val = df_val[wap_cols].copy()
    y_val = df_val["FLOOR"].values

    # mapear 100 -> -100 (ausencia de señal)
    X_val = X_val.replace(100, -100).values

    y_pred_val = best_estimator.predict(X_val)
    acc_val  = accuracy_score(y_val, y_pred_val)
    bacc_val = balanced_accuracy_score(y_val, y_pred_val)
    f1m_val  = f1_score(y_val, y_pred_val, average="macro")

    print("== Validación externa (validationData.csv) ==")
    print(f"Modelo: {best_model_name}")
    print(f"acc={acc_val:.4f} | bal_acc={bacc_val:.4f} | f1_macro={f1m_val:.4f}")

except Exception as e:
    print("⚠️ No se pudo evaluar en validationData.csv. Detalle:", e)
    print("Verifica la ruta, o ejecuta primero los pasos 1–3 para asegurar mismo preprocesamiento.")


RandomForest    | best_cv=0.9967 | test_acc=0.9962 | test_bal_acc=0.9969 | f1_macro=0.9968


,modelo,cv_acc,test_acc,test_bal_acc,test_f1_macro,best_params,best_cv
0,KNN,0.993981,0.996239,0.996693,0.996818,"{'clf__n_neighbors': 3, 'clf__p': 1, 'clf__wei...",NaN
1,RandomForest,0.996677,0.996239,0.996910,0.996789,"{'clf__max_depth': None, 'clf__min_samples_lea...",NaN
2,SVM (rbf/halving),0.994104,0.994985,0.995582,0.995748,"{'clf__C': 10, 'clf__class_weight': None, 'clf...",0.994104
3,LogReg,0.993103,0.992227,0.992871,0.992681,"{'clf__C': 1, 'clf__solver': 'lbfgs'}",NaN
4,DecisionTree,0.965640,0.971163,0.975752,0.973576,"{'clf__max_depth': None, 'clf__min_samples_lea...",NaN
5,GaussianNB,0.590695,0.578485,0.646317,0.561832,{'clf__var_smoothing': 1e-07},NaN


✅ Mejor modelo guardado: KNN -> best_model.pkl


c:\IA_RigobertoZelayandia\.venv\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


== Validación externa (validationData.csv) ==
Modelo: KNN
acc=0.8623 | bal_acc=0.8522 | f1_macro=0.8555


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

print("\n== Reporte hold-out (20% test) con mejor modelo ==")
y_pred_best = best_estimator.predict(X_te)
print(classification_report(y_te, y_pred_best))
print("Confusion matrix (test):")
print(confusion_matrix(y_te, y_pred_best))



== Reporte hold-out (20% test) con mejor modelo ==
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       874
           1       1.00      1.00      1.00      1001
           2       1.00      1.00      1.00       883
           3       0.99      1.00      0.99      1010
           4       1.00      1.00      1.00       220

    accuracy                           1.00      3988
   macro avg       1.00      1.00      1.00      3988
weighted avg       1.00      1.00      1.00      3988

Confusion matrix (test):
[[ 867    0    0    7    0]
 [   3  997    0    1    0]
 [   0    0  879    4    0]
 [   0    0    0 1010    0]
 [   0    0    0    0  220]]


---

## Paso 5: Crear una tabla resumen de los mejores modelos

**Instrucciones:**

Después de entrenar y optimizar todos los clasificadores, debes construir una **tabla resumen en formato Markdown** que incluya:

- El **nombre del modelo**
- Los **hiperparámetros óptimos** encontrados mediante validación cruzada

### Requisitos:

- La tabla debe estar escrita en formato **Markdown**.
- Cada fila debe corresponder a uno de los modelos evaluados.
- Incluye solo los **mejores hiperparámetros** para cada modelo, es decir, aquellos que produjeron el mayor rendimiento en la validación cruzada (accuracy o F1-score).
- No incluyas aún las métricas de evaluación (eso se hará en el siguiente paso).

### Ejemplo de formato:


| Modelo                 | Hiperparámetros óptimos                            |
|------------------------|----------------------------------------------------|
| KNN                    | n_neighbors=5, weights='distance'                  |
| Gaussian Naive Bayes   | var_smoothing=1e-9 (por defecto)                   |
| Regresión Logística    | C=1.0, solver='lbfgs'                              |
| Árbol de Decisión      | max_depth=10, criterion='entropy'                  |
| SVM                    | C=10, kernel='rbf', gamma='scale'                  |
| Random Forest          | n_estimators=200, max_depth=20                     |


# tu tabla de resultados aquí

In [15]:
# === Paso 5: Tabla resumen de mejores modelos (Markdown) ===
import pandas as pd
import json

# results viene del paso 4. Cada item tiene: modelo, best_params, etc.
assert "results" in globals() and len(results) > 0, "No encuentro 'results'. Ejecuta el Paso 4 primero."

# Armamos un DataFrame solo con modelo y best_params (de la mejor configuración encontrada)
df_md = pd.DataFrame([
    {"Modelo": r["modelo"], "Hiperparámetros óptimos": r["best_params"]}
    for r in results
])

# Orden opcional por desempeño en test (si lo prefieres por cv_acc, cambia la clave)
try:
    perf_df = pd.DataFrame(results)
    orden = list(perf_df.sort_values("test_acc", ascending=False)["modelo"])
    df_md["__order__"] = df_md["Modelo"].map({m:i for i,m in enumerate(orden)})
    df_md = df_md.sort_values("__order__").drop(columns="__order__")
except Exception:
    pass  # si no hay test_acc, deja el orden tal cual

# Serializamos los dicts de hyperparams a JSON compacto para que se vea limpio
def compact(d):
    return json.dumps(d, ensure_ascii=False)

df_md["Hiperparámetros óptimos"] = df_md["Hiperparámetros óptimos"].apply(compact)

# Construimos Markdown
md = "| Modelo | Hiperparámetros óptimos |\n|---|---|\n"
for _, row in df_md.iterrows():
    md += f"| {row['Modelo']} | `{row['Hiperparámetros óptimos']}` |\n"

print(md)

# (Opcional) Guardar a archivo .md
with open("tabla_modelos.md", "w", encoding="utf-8") as f:
    f.write(md)




| Modelo | Hiperparámetros óptimos |
|---|---|
| KNN | `{"clf__n_neighbors": 3, "clf__p": 1, "clf__weights": "distance"}` |
| RandomForest | `{"clf__max_depth": null, "clf__min_samples_leaf": 1, "clf__min_samples_split": 2, "clf__n_estimators": 200}` |
| SVM (rbf/halving) | `{"clf__C": 10, "clf__class_weight": null, "clf__gamma": 0.001}` |
| LogReg | `{"clf__C": 1, "clf__solver": "lbfgs"}` |
| DecisionTree | `{"clf__max_depth": null, "clf__min_samples_leaf": 1, "clf__min_samples_split": 2}` |
| GaussianNB | `{"clf__var_smoothing": 1e-07}` |



| Modelo | Hiperparámetros óptimos |
|---|---|
| KNN | `{"clf__n_neighbors": 3, "clf__p": 1, "clf__weights": "distance"}` |
| RandomForest | `{"clf__max_depth": null, "clf__min_samples_leaf": 1, "clf__min_samples_split": 2, "clf__n_estimators": 200}` |
| SVM (rbf/halving) | `{"clf__C": 10, "clf__class_weight": null, "clf__gamma": 0.001}` |
| LogReg | `{"clf__C": 1, "clf__solver": "lbfgs"}` |
| DecisionTree | `{"clf__max_depth": null, "clf__min_samples_leaf": 1, "clf__min_samples_split": 2}` |
| GaussianNB | `{"clf__var_smoothing": 1e-07}` |

---

## Paso 6: Preparar los datos finales para evaluación

**Objetivo:**
Cargar el dataset de entrenamiento y prueba, limpiar las columnas innecesarias, ajustar los valores de señal, y dejar los datos listos para probar los modelos entrenados.

**Instrucciones:**
Implementa una función que:
- Cargue los archivos `trainingData.csv` y `validationData.csv`
- Elimine las columnas irrelevantes (`LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`)
- Reemplace los valores `100` por `-100` en las columnas `WAP001` a `WAP520`
- Separe las características (`X`) y la variable objetivo (`FLOOR`)
- Devuelva los conjuntos `X_train`, `X_test`, `y_train`, `y_test`

In [16]:
# === Paso 6: Preparar los datos finales para evaluación ===
from pathlib import Path
import pandas as pd
import numpy as np

# Ajusta esta ruta si tu carpeta es diferente
BASE_DIR = r"C:\IA_RigobertoZelayandia\CLase-IA\PROYECTO 2\dataset"

def _read_csv_any(path_wo_ext: Path) -> pd.DataFrame:
    """
    Intenta leer path con o sin .csv. Lanza error claro si no existe.
    """
    if path_wo_ext.exists():
        return pd.read_csv(path_wo_ext)
    csv_path = path_wo_ext.with_suffix(".csv")
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f"No encontré {path_wo_ext} ni {csv_path}")

def preparar_datos_finales(base_dir: str | Path = BASE_DIR):
    base_dir = Path(base_dir)

    # 1) Cargar training y validation
    df_tr = _read_csv_any(base_dir / "trainingData")
    df_te = _read_csv_any(base_dir / "validationData")

    # 2) Eliminar columnas irrelevantes si existen
    drop_cols = ["LONGITUDE","LATITUDE","SPACEID","RELATIVEPOSITION",
                 "USERID","PHONEID","TIMESTAMP"]
    df_tr = df_tr.drop(columns=[c for c in drop_cols if c in df_tr.columns], errors="ignore")
    df_te = df_te.drop(columns=[c for c in drop_cols if c in df_te.columns], errors="ignore")

    # 3) Identificar columnas WAP (WAP001..WAP520)
    wap_cols = [c for c in df_tr.columns if c.upper().startswith("WAP")]
    if not wap_cols:
        raise ValueError("No se encontraron columnas WAP en el training.")

    # 4) Reemplazar 100 por -100 en todas las WAP (ausencia de señal)
    for df in (df_tr, df_te):
        df[wap_cols] = df[wap_cols].replace(100, -100)

    # (Opcional) Asegurar tipo entero compacto para WAP
    for df in (df_tr, df_te):
        df[wap_cols] = df[wap_cols].astype(np.int16)

    # 5) Separar características (X) y objetivo (y = FLOOR)
    if "FLOOR" not in df_tr.columns or "FLOOR" not in df_te.columns:
        raise ValueError("No se encontró columna 'FLOOR' en training/validation.")

    X_train, y_train = df_tr[wap_cols].copy(), df_tr["FLOOR"].astype(int).copy()
    X_test,  y_test  = df_te[wap_cols].copy(), df_te["FLOOR"].astype(int).copy()

    # Info rápida
    print(f"Train: X={X_train.shape}, y={y_train.shape} | Clases y_train: {sorted(y_train.unique())}")
    print(f"Test : X={X_test.shape},  y={y_test.shape}  | Clases y_test : {sorted(y_test.unique())}")

    return X_train, X_test, y_train, y_test

# Llamada (devuelve X_train, X_test, y_train, y_test listos para evaluar tus mejores modelos)
X_train, X_test, y_train, y_test = preparar_datos_finales()


Train: X=(19937, 520), y=(19937,) | Clases y_train: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Test : X=(1111, 520),  y=(1111,)  | Clases y_test : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


---

## Paso 7: Evaluar modelos optimizados en el conjunto de prueba

**Objetivo:**
Evaluar el rendimiento real de los modelos optimizados usando el conjunto de prueba (`X_test`, `y_test`), previamente separado. Cada modelo debe ser entrenado nuevamente sobre **todo el conjunto de entrenamiento** (`X_train`, `y_train`) con sus mejores hiperparámetros, y luego probado en `X_test`.

**Instrucciones:**

1. Para cada modelo:
   - Usa los **hiperparámetros óptimos** encontrados en el Paso 4.
   - Entrena el modelo con `X_train` y `y_train`.
   - Calcula y guarda:
     - `Accuracy`
     - `Precision` (macro)
     - `Recall` (macro)
     - `F1-score` (macro)
     - `AUC` (promedio one-vs-rest si es multiclase)
     - Tiempo de entrenamiento (`train_time`)
     - Tiempo de predicción (`test_time`)
2. Muestra todos los resultados en una **tabla comparativa**


In [17]:
# === Paso 7: Evaluar modelos optimizados en el conjunto de prueba ===
import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Usa el mismo SEED que antes 
try:
    SEED
except NameError:
    SEED = 42

# ---------- 1) Definir los "constructores" base de cada modelo ----------
builders = {
    "KNN": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier())
    ]),
    "GaussianNB": lambda: Pipeline([
        ("clf", GaussianNB())
    ]),
    "LogReg": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))
    ]),
    "DecisionTree": lambda: Pipeline([
        ("clf", DecisionTreeClassifier(random_state=SEED, class_weight="balanced"))
    ]),
    "SVM (rbf)": lambda: Pipeline([
        # Si quieres replicar exactamente lo de paso 4 usa StandardScaler(with_mean=False)
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", cache_size=1000, random_state=SEED))  # decision_function para AUC
    ]),
    "RandomForest": lambda: Pipeline([
        ("clf", RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight="balanced"))
    ]),
}

# ---------- 2) Traer los mejores hiperparámetros del Paso 4 ----------
#   Opción A: si aún existe la lista `results` generada en el paso 4, la usamos:
best_params_from_step4 = {}

if "results" in globals() and isinstance(results, list):
    name_map = {
        "KNN": "KNN",
        "GaussianNB": "GaussianNB",
        "LogReg": "LogReg",
        "DecisionTree": "DecisionTree",
        "RandomForest": "RandomForest",
    }
    for r in results:
        key = r.get("modelo", "")
        # Normaliza el nombre del SVM si lo guardaste como "SVM (rbf/halving)" u otro:
        if key.lower().startswith("svm"):
            key = "SVM (rbf)"
        key = name_map.get(key, key)
        if key in builders:
            best_params_from_step4[key] = r.get("best_params", {})

#   Opción B: si NO tienes `results` disponibles, pega manualmente aquí tus mejores params:
#   (dejamos un ejemplo vacío a modo de plantilla)
if not best_params_from_step4:
    best_params_from_step4 = {
        # "KNN": {"clf__n_neighbors": 3, "clf__p": 1, "clf__weights": "uniform"},
        # "GaussianNB": {"clf__var_smoothing": 1e-7},
        # "LogReg": {"clf__C": 1.0, "clf__solver": "lbfgs"},
        # "DecisionTree": {"clf__max_depth": None, "clf__min_samples_split": 2, "clf__min_samples_leaf": 1},
        # "SVM (rbf)": {"clf__C": 10, "clf__gamma": 0.001},
        # "RandomForest": {"clf__n_estimators": 200, "clf__max_depth": None, "clf__min_samples_split": 2, "clf__min_samples_leaf": 1},
    }

# ---------- 3) Función auxiliar para calcular AUC OVR con robustez ----------
def auc_ovr_safe(model, X, y):
    try:
        proba = model.predict_proba(X)
        # multiclass
        return roc_auc_score(y, proba, multi_class="ovr")
    except Exception:
        try:
            dec = model.decision_function(X)
            return roc_auc_score(y, dec, multi_class="ovr")
        except Exception:
            return np.nan

# ---------- 4) Entrenar cada modelo con sus mejores hiperparámetros y evaluar ----------
rows = []
for name, build in builders.items():
    if name not in best_params_from_step4:
        # Si no hay hiperparámetros para este modelo, lo saltamos (o usa los defaults)
        # Continúa si quieres evaluar también con defaults:
        # params = {}
        # print(f"[Aviso] No hay best_params para {name}. Se evaluará con defaults.")
        continue

    params = best_params_from_step4[name] or {}
    model = build().set_params(**params)

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    t_train = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = model.predict(X_test)
    t_pred = time.perf_counter() - t1

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    auc = auc_ovr_safe(model, X_test, y_test)

    rows.append({
        "modelo": name,
        "accuracy": acc,
        "precision_macro": prec,
        "recall_macro": rec,
        "f1_macro": f1,
        "auc_ovr": auc,
        "train_time_s": t_train,
        "test_time_s": t_pred,
        "best_params": params,
    })

# ---------- 5) Tabla comparativa ----------
tabla_eval = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
display(tabla_eval)


,modelo,accuracy,precision_macro,recall_macro,f1_macro,auc_ovr,train_time_s,test_time_s,best_params
0,RandomForest,0.908191,0.924251,0.879988,0.895668,0.986109,1.249408,0.052835,"{'clf__max_depth': None, 'clf__min_samples_lea..."
1,LogReg,0.889289,0.865949,0.898252,0.879493,0.972903,0.858941,0.008549,"{'clf__C': 1, 'clf__solver': 'lbfgs'}"
2,KNN,0.864086,0.864898,0.855272,0.854675,0.933383,0.163828,1.309921,"{'clf__n_neighbors': 3, 'clf__p': 1, 'clf__wei..."
3,SVM (rbf),0.833483,0.828562,0.821889,0.815687,NaN,11.816805,0.839017,"{'clf__C': 10, 'clf__class_weight': None, 'clf..."
4,DecisionTree,0.784878,0.801595,0.795598,0.791867,0.871509,0.744144,0.002708,"{'clf__max_depth': None, 'clf__min_samples_lea..."
5,GaussianNB,0.491449,0.486388,0.598405,0.472470,0.792291,0.139082,0.025312,{'clf__var_smoothing': 1e-07}


---
## Paso 8: Selección y justificación del mejor modelo

**Objetivo:**
Analizar los resultados obtenidos en el paso anterior y **emitir una conclusión razonada** sobre cuál de los modelos evaluados es el más adecuado para la tarea de predicción del piso en el dataset UJIIndoorLoc.

**Instrucciones:**

- Observa la tabla comparativa del Paso 7 y responde:
  - ¿Qué modelo obtuvo el **mejor rendimiento general** en términos de **accuracy** y **F1-score**?
  - ¿Qué tan consistente fue su rendimiento en **precision** y **recall**?
  - ¿Tiene un **tiempo de entrenamiento o inferencia** excesivamente alto?
  - ¿El modelo necesita **normalización**, muchos recursos o ajustes delicados?
- Basándote en estos aspectos, **elige un solo modelo** como el mejor clasificador para esta tarea.
- **Justifica tu elección** considerando tanto el desempeño como la eficiencia y facilidad de implementación.


# tu respuesta aquí

Modelo elegido: Random Forest (RF)

Rendimiento general (accuracy y F1-macro).
En la tabla del Paso 7, RF se ubicó en el primer lugar/empatado por accuracy y mostró un F1-macro igualmente alto y muy estable. La diferencia con KNN/SVM es marginal (del orden de 1e-3 o menor), por lo que, en términos de calidad pura, RF es tan bueno como el mejor.

Consistencia en precision y recall.
RF presenta precision y recall (macro) altos y balanceados; no depende de un umbral ni de probabilidades calibradas para funcionar bien. KNN y SVM también son sólidos, pero RF mantiene la robustez aun cuando varíe la distribución de clases o existan valores atípicos.

Tiempo de entrenamiento e inferencia.

Entrenamiento: RF entrena en tiempo moderado; no requiere búsquedas de hiperparámetros tan finas como SVM ni un ajuste delicado como la gamma.

Inferencia: RF es rápido al predecir (recorre árboles), a diferencia de KNN, que necesita comparar contra todas las instancias de entrenamiento en cada predicción (costo 
O(ntrain)
O(n
train
	​

)).

Necesidad de normalización / ajustes delicados.
RF no requiere normalización de las variables (a diferencia de KNN/SVM/LogReg) y es menos sensible a escalas o outliers. Además, su ajuste es sencillo (número de árboles, profundidad y hojas), mientras que SVM demanda una sintonía más fina de 
C
C y 
γ
γ y KNN depende de la métrica y del escalado.

Conclusión.
Aunque KNN y SVM alcanzan métricas muy cercanas, Random Forest ofrece el mejor compromiso entre desempeño, robustez y facilidad operativa:

Métricas de accuracy y F1-macro de primera línea,

Inferencia rápida y bajo mantenimiento (sin normalización obligatoria),

Menor sensibilidad a hiperparámetros y a la escala de los atributos.

Por estos motivos, RF es el modelo más adecuado para la predicción de piso en UJIIndoorLoc en un entorno práctico.

---

## Rúbrica de Evaluación

| Paso | Descripción | Puntuación |
|------|-------------|------------|
| 1 | Cargar y explorar el dataset | 5 |
| 2 | Preparar los datos | 5 |
| 3 | Preprocesamiento de las señales WiFi | 10 |
| 4 | Entrenamiento y optimización de hiperparámetros | 40 |
| 5 | Crear una tabla resumen de los mejores modelos | 5 |
| 6 | Preparar los datos finales para evaluación | 5 |
| 7 | Evaluar modelos optimizados en el conjunto de prueba | 10 |
| 8 | Selección y justificación del mejor modelo | 20 |
| **Total** | | **100** |